# 02 — PySpark Lakehouse Processing and CDC

## Business scenario

Bronze now contains immutable API envelopes and compressed application logs. We
need analytics-ready Silver tables without silently discarding malformed records.
We also receive change-data-capture events for order updates and deletes.

### Learning objectives

- Read nested JSON with an explicit Spark schema.
- Handle schema evolution from `customer_id` to `customer.id`.
- Normalize an order array into order and order-item tables.
- Quarantine invalid records with reasons and source metadata.
- Deduplicate at-least-once source delivery deterministically.
- Write Hive-style partitioned Parquet.
- Resolve out-of-order CDC events and prove idempotency.
- Translate the rules to Delta Lake `MERGE` on Databricks.

Run Notebook 01 first. Local PySpark requires Java 17+ and the packages in
`requirements-spark.txt`.


## What is CDC?

CDC stands for **Change Data Capture**. Instead of loading the entire source table every time, CDC delivers only inserted, updated, or deleted records.

```text
I (Insert) → 새로운 주문 추가
U (Update) → 기존 주문 변경
D (Delete) → 기존 주문 삭제
```

Example:

```text
ord_001 | I | pending → 새 주문 생성
ord_001 | U | paid    → 결제 상태로 변경
ord_002 | D |         → 주문 삭제
```

## Notebook Flow

```text
Notebook 01: Bronze Data
├── Raw API order JSON
└── Compressed application logs
              │
              ▼
PySpark reads data with explicit schemas
              │
              ▼
Parse and normalize
├── 서로 다른 schema version 통합
├── 문자열을 숫자와 timestamp로 변환
└── 중첩된 items와 log context 펼치기
              │
              ▼
Data-quality checks
├── 정상 데이터
└── 잘못된 데이터 → Quarantine
              │
              ▼
Deduplication and transformationormalization
├── 중복 주문 제거
├── Orders 테이블 생성
└── Order Items 테이블 생성
              │
              ▼
Silver Parquet
├── silver/orders
├── silver/order_items
└── silver/application_logs
              │
              ▼
Apply CDC Events
├── I: 주문 추가
├── U: 주문 변경
└── D: 주문 삭제
              │
              ▼
Current Order Snapshot
현재 시점의 최종 주문 상태
              │
              ▼
Run CDC again
동일한 결과 유지 → Idempotency 확인
```

## Main Goal

This notebook transforms raw and nested Bronze data into clean, tabular, analytics-ready Silver datasets. It also applies CDC events to maintain the latest state of each order without rebuilding the entire dataset.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

LAB_ROOT = PROJECT_ROOT / "lab_data"
BRONZE_API = LAB_ROOT / "bronze" / "api" / "orders"
BRONZE_LOGS = LAB_ROOT / "bronze" / "logs"
SILVER_ROOT = LAB_ROOT / "silver"
ORDERS_PATH = SILVER_ROOT / "orders"
ITEMS_PATH = SILVER_ROOT / "order_items"
QUARANTINE_PATH = SILVER_ROOT / "quarantine_orders"
LOGS_PATH = SILVER_ROOT / "application_logs"
CDC_PATH = LAB_ROOT / "bronze" / "cdc" / "orders_cdc.jsonl"

raw_pages = sorted(BRONZE_API.glob("ingestion_date=*/run_id=*/page_*.json"))
if not raw_pages:
    raise FileNotFoundError("No Bronze API pages found. Run notebook 01 first.")
print(f"Discovered {len(raw_pages)} raw API pages")


Discovered 7 raw API pages


## Start Spark intentionally

`local[*]` uses multiple threads on one machine. It is valuable for learning Spark
execution, partitions, lazy evaluation, and shuffle behavior, but it is **not** a
multi-node cluster and should not be described as EMR operations experience.
```text
local[*]:

현재 컴퓨터(예를 들어 컴퓨터에 CPU 논리 코어가 8개 있다면):
├── CPU thread 1 → Spark task 처리
├── CPU thread 2 → Spark task 처리
├── CPU thread 3 → Spark task 처리
├── ...
└── CPU thread 8 → Spark task 처리
```
- local[*]는 한 컴퓨터의 여러 CPU thread로 Spark의 분산 처리 방식을 연습하는 설정이며, Spark 개발 학습에는 유용하지만 실제 여러 서버나 AWS EMR을 운영한 경험과는 다릅니다.

## AWS Data Architecture Patterns — Horizontal Comparison

| 기본 구조 | EMR 중심 Data Lake | Redshift 중심 Warehouse | Hybrid 구조 |
|:---:|:---:|:---:|:---:|
| **API / Files**<br>↓<br>**Ingestion Pipeline**<br>↓<br>**S3 Data Lake**<br>↓<br>**Athena** | **API / Files**<br>↓<br>**S3 Bronze**<br>↓<br>**EMR + PySpark**<br>↓<br>**S3 Silver / Gold**<br>↓<br>**Athena** | **API / Database**<br>↓<br>**ETL / ELT**<br>↓<br>**Redshift**<br>↓<br>**Data Mart**<br>↓<br>**BI Dashboard** | **API / Database**<br>↓<br>**S3 Bronze**<br>↓<br>**EMR + PySpark**<br>↓<br>**S3 Curated**<br>↙　↘<br>**Athena　Redshift**<br>↓<br>**BI** |
| 원본 및 비정형 데이터를 S3에서 바로 조회 | Spark로 대용량 데이터를 정제하고 다시 S3에 저장 | 정형 테이블과 반복적인 BI 분석에 집중 | Data Lake의 유연성과 Warehouse의 분석 성능을 함께 사용 |
| **Athena:** ad-hoc SQL | **EMR:** 대규모 ETL | **Redshift:** Data Warehouse | **Athena + EMR + Redshift** |

In [2]:
try:
    from pyspark.sql import SparkSession, Window
    from pyspark.sql import functions as F
    from pyspark.sql import types as T
except ImportError as exc:
    raise RuntimeError(
        "PySpark is not installed. Run: python -m pip install -r requirements-spark.txt"
    ) from exc

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("commerce-lakehouse-cdc")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(spark.version)


4.2.0


## Use an explicit source schema

Schema inference scans data and can make inconsistent choices when fields contain
mixed types. A production ingestion contract should declare expected fields and
preserve unknown or invalid values for investigation.

`total_amount` is read as a string first. Casting belongs in transformation so a
value such as `"not-a-number"` can be quarantined rather than turned into an
unexplained null.


```text
Bronze page JSON files
        ↓
envelope_schema 적용
├── data: 주문 배열
├── next_cursor: 다음 페이지 위치
├── request_id: API 요청 추적 ID
└── schema: API 응답 구조 버전
        ↓
각 주문에 order_schema 적용
├── customer: 중첩된 고객 정보
└── items: 상품 배열
        │
        ├── sku
        ├── quantity
        └── unit_price
        ↓
Spark가 여러 JSON 파일 읽기
        ↓
envelopes
한 행 = API 페이지 하나
        ↓
source_file 컬럼 추가
어떤 Bronze 파일에서 왔는지 기록
        ↓
data 배열을 explode_outer()
        ↓
orders_raw
한 행 = 주문 하나
├── order
├── request_id
└── source_file
        ↓
printSchema()
최종 중첩 구조 확인
```

In [3]:
# 한 주문에 포함된 상품 한 개의 구조를 정의한다.
item_schema = T.StructType([
    T.StructField("sku", T.StringType()),
    T.StructField("quantity", T.LongType()),
    T.StructField("unit_price", T.StringType()),
])

# Schema version 2에서 사용하는 중첩 customer 구조를 정의한다.
customer_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("tier", T.StringType()),
])

# 주문 한 건의 전체 구조를 정의한다.
order_schema = T.StructType([
    T.StructField("order_id", T.StringType()),

    # Version 1의 고객 ID
    T.StructField("customer_id", T.StringType()),

    # Version 2의 중첩 고객 정보
    T.StructField("customer", customer_schema),

    T.StructField("event_type", T.StringType()),
    T.StructField("event_timestamp", T.StringType()),
    T.StructField("updated_at", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("currency", T.StringType()),

    # 잘못된 값도 우선 읽을 수 있도록 문자열로 정의한다.
    T.StructField("total_amount", T.StringType()),

    # 한 주문에는 여러 상품이 들어갈 수 있다.
    T.StructField("items", T.ArrayType(item_schema)),

    T.StructField("schema_version", T.LongType()),
])

# API가 반환하는 바깥쪽 JSON envelope 구조를 정의한다.
envelope_schema = T.StructType([
    T.StructField("data", T.ArrayType(order_schema)),
    T.StructField("next_cursor", T.StringType()),
    T.StructField("request_id", T.StringType()),
    T.StructField("schema", T.StringType()),
])


# Path 객체를 Spark가 읽을 수 있는 문자열 목록으로 변환한다.
page_names = [
    str(path)
    for path in raw_pages
]

# 여러 Bronze JSON 페이지를 명시적인 schema로 읽는다.
envelopes = (
    spark.read
    .schema(envelope_schema)

    # 하나의 JSON 객체가 여러 줄로 저장되어 있음을 알려준다.
    .option("multiLine", True)

    .json(page_names)

    # 각 데이터가 어느 Bronze 파일에서 왔는지 기록한다.
    .withColumn(
        "source_file",
        F.input_file_name(),
    )
)

# data 배열을 펼쳐 주문 하나가 한 행이 되도록 만든다.
orders_raw = envelopes.select(
    F.explode_outer("data").alias("order"),
    "request_id",
    "source_file",
)

# Spark가 읽은 최종 중첩 schema를 확인한다.
orders_raw.printSchema()


root
 |-- order: struct (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- customer: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- tier: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- updated_at: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- total_amount: string (nullable = true)
 |    |-- items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- sku: string (nullable = true)
 |    |    |    |-- quantity: long (nullable = true)
 |    |    |    |-- unit_price: string (nullable = true)
 |    |-- schema_version: long (nullable = true)
 |-- request_id: string (nullable = true)
 |-- source_file: string (nullable = false)



```text
orders_raw
   ↓
필드 선택 + schema version 통합
   ↓
timestamp와 금액 타입 변환
   ↓
normalized
   ↓
데이터 품질 검사
   ├── 문제 있음 → quarantine
   └── 문제 없음 → candidates
```

In [4]:
normalized = orders_raw.select(
    F.col("order.order_id").alias("order_id"),
    F.coalesce(F.col("order.customer_id"), F.col("order.customer.id")).alias("customer_id"),
    F.col("order.event_type").alias("event_type"),
    F.to_timestamp("order.event_timestamp").alias("event_timestamp"),
    F.to_timestamp("order.updated_at").alias("updated_at"),
    F.col("order.status").alias("status"),
    F.col("order.currency").alias("currency"),
    F.col("order.total_amount").alias("raw_total_amount"),
    F.col("order.total_amount").try_cast("decimal(18,2)").alias("total_amount"),
    F.col("order.items").alias("items"),
    F.col("order.schema_version").alias("schema_version"),
    "request_id",
    "source_file",
).withColumn("event_date", F.to_date("event_timestamp"))

reason = (
    F.when(F.col("order_id").isNull(), F.lit("missing_order_id"))
    .when(F.col("customer_id").isNull(), F.lit("missing_customer_id"))
    .when(F.col("event_timestamp").isNull(), F.lit("invalid_event_timestamp"))
    .when(F.col("currency").isNull(), F.lit("missing_currency"))
    .when(F.col("total_amount").isNull(), F.lit("invalid_total_amount"))
    .when(F.size(F.coalesce(F.col("items"), F.array())) == 0, F.lit("missing_items"))
)
classified = normalized.withColumn("quarantine_reason", reason)
quarantine = classified.filter(F.col("quarantine_reason").isNotNull())
candidates = classified.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")

print("Raw rows:", normalized.count())
print("Quarantined rows:", quarantine.count())
quarantine.groupBy("quarantine_reason").count().orderBy("quarantine_reason").show(truncate=False)


Raw rows: 241
Quarantined rows: 2
+--------------------+-----+
|quarantine_reason   |count|
+--------------------+-----+
|invalid_total_amount|1    |
|missing_currency    |1    |
+--------------------+-----+



## Deterministic deduplication

`dropDuplicates("order_id")` does not express which duplicate wins. We define a
stable ordering: latest source `updated_at`, then source file. This matters because
Spark task order is not a business rule.


```text
주문 데이터
   ↓
order_id별 최신 주문 선택
   ├── 1순위 → valid_orders
   └── 2순위 이상 → duplicates
                   　
valid_orders의 items 배열
   ↓
상품 하나당 한 행으로 펼치기
   ↓
valid_order_items
                   　
전체 건수
= 정상 주문 + 격리 주문 + 중복 주문
```

In [6]:
# order_id가 같은 데이터끼리 묶은 후 가장 최신 주문을 1순위로
dedupe_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        # updated_at이 가장 최신인 주문을 우선 선택한다.
        F.col("updated_at").desc_nulls_last(),

        # updated_at이 같으면 source_file 이름으로 순서를 결정한다.
        F.col("source_file").desc(),
    )
)

# 각 order_id 안에서 최신 순서대로 1, 2, 3 등의 순위를 부여한다.
ranked = candidates.withColumn(
    "dedupe_rank",
    F.row_number().over(dedupe_window),
)

# 2순위 이후의 데이터는 동일한 order_id의 중복 데이터이다.
duplicates = ranked.filter(
    F.col("dedupe_rank") > 1
)

# 가장 최신인 1순위 주문만 정상 주문 데이터로 선택한다.
valid_orders = (
    ranked
    .filter(F.col("dedupe_rank") == 1)
    .drop("dedupe_rank")
)

# 한 주문 안의 items 배열을 여러 개의 상품 행으로 펼친다.
valid_order_items = (
    valid_orders

    # posexplode는 배열의 위치와 각 상품을 별도의 행으로 만든다.
    .select(
        "order_id",
        "event_date",
        F.posexplode("items").alias(
            "line_offset",
            "item",
        ),
    )

    # 중첩된 item 필드를 분석 가능한 일반 컬럼으로 변환한다.
    .select(
        "order_id",
        "event_date",

        # 배열 위치는 0부터 시작하므로 1을 더해 상품 순번을 만든다.
        (F.col("line_offset") + 1).alias(
            "line_number"
        ),

        F.col("item.sku").alias("sku"),

        # 상품 수량을 정수형으로 변환한다.
        F.col("item.quantity")
        .cast("long")
        .alias("quantity"),

        # 상품 가격을 소수점 둘째 자리까지의 숫자로 안전하게 변환한다.
        # 변환할 수 없는 값은 오류 대신 null이 된다.
        F.col("item.unit_price")
        .try_cast("decimal(18,2)")
        .alias("unit_price"),
    )
)


# 데이터 품질과 변환 결과를 숫자로 요약한다.
quality_metrics = {
    # Bronze에서 읽고 구조를 통합한 전체 주문 수
    "input_count": normalized.count(),

    # 데이터 품질 검사를 통과하고 중복이 제거된 주문 수
    "valid_order_count": valid_orders.count(),

    # 필수값 누락이나 잘못된 데이터 타입으로 격리된 주문 수
    "quarantine_count": quarantine.count(),

    # 동일한 order_id로 발견된 중복 주문 수
    "duplicate_count": duplicates.count(),

    # items 배열을 행으로 펼친 후 생성된 전체 상품 행 수
    "item_count": valid_order_items.count(),
}


# 입력 주문이 정상, 격리, 중복 중 하나로 빠짐없이 분류됐는지 확인한다.
assert quality_metrics["input_count"] == (
    quality_metrics["valid_order_count"]
    + quality_metrics["quarantine_count"]
    + quality_metrics["duplicate_count"]
)

# 최종 데이터 품질 지표를 출력한다.
quality_metrics


{'input_count': 241,
 'valid_order_count': 238,
 'quarantine_count': 2,
 'duplicate_count': 1,
 'item_count': 608}

## Write Silver tables with useful partitions

We partition orders by `event_date`, a low-cardinality field commonly used by
incremental jobs and date-range queries. We do **not** partition by high-cardinality
`order_id`; that would create excessive directories and small files.

`mode("overwrite")` is acceptable for this controlled rebuild lesson. A production
job should overwrite only affected partitions or use a transactional table format.


```text
이전 Silver 결과 삭제
        ↓
Spark DataFrame
├── valid_orders
├── valid_order_items
└── quarantine
        ↓
작은 실습 데이터를 Pandas로 변환
        ↓
PyArrow로 로컬 파일 저장
├── Silver orders     → 날짜별 Parquet
├── Silver items      → 날짜별 Parquet
└── Quarantine orders → JSONL
        ↓
생성된 날짜 partition 확인
```

In [7]:
import shutil


def reset_output(path: Path) -> None:
    """기존 Silver 실습 결과만 안전하게 삭제한다."""

    if path.exists():
        # 실수로 프로젝트 밖의 폴더를 삭제하지 않도록 경로를 확인한다.
        assert SILVER_ROOT in path.parents

        # 이전 실행 결과를 제거하여 새 결과와 섞이지 않게 한다.
        shutil.rmtree(path)


# 이전 실행에서 생성된 Silver 결과를 삭제한다.
for output_path in (
    ORDERS_PATH,
    ITEMS_PATH,
    QUARANTINE_PATH,
):
    reset_output(output_path)


# 데이터 정제, 중복 제거, 배열 변환은 앞에서 Spark가 수행했다.
#
# Windows에서는 Spark의 로컬 Parquet 쓰기 문제가 발생할 수 있으므로
# 최종 파일 저장 부분만 Pandas와 PyArrow에 맡긴다.
orders_pdf = (
    valid_orders
    .drop("items")
    .toPandas()
)

items_pdf = valid_order_items.toPandas()


# event_date를 문자열로 변환해 날짜별 partition 폴더 이름으로 사용한다.
#
# 예:
# event_date=2026-01-01/
# event_date=2026-01-02/
orders_pdf["event_date"] = (
    orders_pdf["event_date"].astype(str)
)

items_pdf["event_date"] = (
    items_pdf["event_date"].astype(str)
)


# 정상 주문을 날짜별로 partition하여 Parquet으로 저장한다.
orders_pdf.to_parquet(
    ORDERS_PATH,
    partition_cols=["event_date"],
    index=False,
)


# 주문 상품도 날짜별로 partition하여 Parquet으로 저장한다.
items_pdf.to_parquet(
    ITEMS_PATH,
    partition_cols=["event_date"],
    index=False,
)


# 잘못된 주문을 보관할 quarantine 폴더를 만든다.
QUARANTINE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)


# 데이터 품질 검사를 통과하지 못한 주문을 JSONL로 저장한다.
# 원본 문제를 조사하거나 수정 후 재처리할 때 사용할 수 있다.
quarantine.toPandas().to_json(
    QUARANTINE_PATH / "part-00000.jsonl",
    orient="records",
    lines=True,
    date_format="iso",
)


print(
    "Windows fallback: "
    "PyArrow wrote the Silver outputs."
)


# 실제로 생성된 주문 날짜 partition을 확인한다.
print("Order partitions:")

for path in sorted(
    ORDERS_PATH.glob("event_date=*")
):
    print(" -", path.name)

Windows fallback: PyArrow wrote the Silver outputs.
Order partitions:
 - event_date=2026-01-01
 - event_date=2026-01-02
 - event_date=2026-01-03


## Parse nested application logs and preserve corrupt input

A malformed log line must not terminate the entire batch, but it also must not
disappear. Spark's permissive parser can retain the original line in a corrupt
record column. That record goes to quarantine with source metadata.


In [8]:
log_schema = T.StructType([
    T.StructField("event_id", T.StringType()),
    T.StructField("timestamp", T.StringType()),
    T.StructField("service", T.StringType()),
    T.StructField("level", T.StringType()),
    T.StructField("trace_id", T.StringType()),
    T.StructField("message", T.StringType()),
    T.StructField("context", T.StructType([
        T.StructField("customer_id", T.StringType()),
        T.StructField("latency_ms", T.LongType()),
        T.StructField("http", T.StructType([T.StructField("status", T.LongType())])),
    ])),
    T.StructField("exception", T.StructType([
        T.StructField("type", T.StringType()),
        T.StructField("stack_trace", T.StringType()),
    ])),
    T.StructField("_corrupt_record", T.StringType()),
])
log_files = [str(path) for path in BRONZE_LOGS.glob("event_date=*/*.jsonl.gz")]
logs_raw = (
    spark.read.schema(log_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(log_files)
    .withColumn("source_file", F.input_file_name())
).cache()
logs_raw.count()  # Materialize before querying the corrupt-record column alone.
clean_logs = logs_raw.filter(F.col("_corrupt_record").isNull()).select(
    "event_id",
    F.to_timestamp("timestamp").alias("event_timestamp"),
    "service", "level", "trace_id", "message",
    F.col("context.customer_id").alias("customer_id"),
    F.col("context.latency_ms").alias("latency_ms"),
    F.col("context.http.status").alias("http_status"),
    F.col("exception.type").alias("exception_type"),
    F.col("exception.stack_trace").alias("stack_trace"),
    "source_file",
).withColumn("event_date", F.to_date("event_timestamp"))
corrupt_logs = logs_raw.filter(F.col("_corrupt_record").isNotNull())

# clean_logs.write.mode("overwrite").partitionBy("event_date").parquet(str(LOGS_PATH))

reset_output(LOGS_PATH)

clean_logs_pdf = clean_logs.toPandas()
clean_logs_pdf["event_date"] = clean_logs_pdf["event_date"].astype(str)

clean_logs_pdf.to_parquet(
    LOGS_PATH,
    partition_cols=["event_date"],
    index=False,
)
print("Clean logs:", clean_logs.count(), "Corrupt logs:", corrupt_logs.count())
assert corrupt_logs.count() == 1


Clean logs: 250 Corrupt logs: 1


## Change data capture: define ordering before code

Each event has an operation (`I`, `U`, `D`) and a monotonically increasing source
`sequence_no`. Arrival order is deliberately scrambled. For each `order_id`, the
greatest sequence is authoritative. Deletes remove the current row.

Event time alone is not a safe tiebreaker. Database log sequence numbers or source
offsets are stronger when available.


```text
valid_orders
현재 정상 주문
     ↓
12개 주문을 seed_orders로 선택
     ↓
합성 CDC event 생성
├── U: 기존 주문 상태 변경
├── D: 기존 주문 삭제
├── I: 새로운 주문 추가
├── 과거 event가 늦게 도착
└── 동일 event가 중복 도착
     ↓
Bronze CDC JSONL 파일로 저장
     ↓
Spark가 명시적인 schema로 읽기
     ↓
changes DataFrame 생성
```

In [9]:
seed_orders = valid_orders.select(
    "order_id", "customer_id", "status", "currency", "total_amount", "updated_at"
).limit(12)
seed_ids = [row.order_id for row in seed_orders.select("order_id").collect()]

cdc_events = [
    {"order_id": seed_ids[0], "op": "U", "sequence_no": 102, "status": "shipped", "total_amount": "88.00"},
    {"order_id": seed_ids[0], "op": "U", "sequence_no": 101, "status": "paid", "total_amount": "88.00"},
    {"order_id": seed_ids[1], "op": "D", "sequence_no": 103, "status": None, "total_amount": None},
    {"order_id": "ord_999999", "op": "I", "sequence_no": 104, "status": "paid", "total_amount": "42.50"},
    {"order_id": "ord_999999", "op": "I", "sequence_no": 104, "status": "paid", "total_amount": "42.50"},
]
CDC_PATH.parent.mkdir(parents=True, exist_ok=True)
with CDC_PATH.open("w", encoding="utf-8") as destination:
    for event in cdc_events:
        destination.write(json.dumps(event) + "\n")

cdc_schema = "order_id string, op string, sequence_no long, status string, total_amount decimal(18,2)"
changes = spark.read.schema(cdc_schema).json(str(CDC_PATH))
changes.orderBy("sequence_no", "order_id").show()


+----------+---+-----------+-------+------------+
|  order_id| op|sequence_no| status|total_amount|
+----------+---+-----------+-------+------------+
|ord_000001|  U|        101|   paid|       88.00|
|ord_000001|  U|        102|shipped|       88.00|
|ord_000002|  D|        103|   NULL|        NULL|
|ord_999999|  I|        104|   paid|       42.50|
|ord_999999|  I|        104|   paid|       42.50|
+----------+---+-----------+-------+------------+



```text
CDC changes
    ↓
order_id별 최신 sequence_no 선택
    ↓
latest
├── op = D     → 삭제 대상
└── op = I/U   → 추가 또는 변경 대상
                     　
현재 snapshot
    ↓
Delete 대상 제거
    ↓
survivors
    ↓
Insert/Update 대상의 기존 행 제거
    ↓
변경되지 않은 주문 + 최신 Insert/Update
    ↓
최종 주문 snapshot
```

In [10]:
def apply_cdc(current_df, change_df):
    latest = (
        change_df
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("order_id").orderBy(F.col("sequence_no").desc())
            ),
        )
        .filter("rn = 1")
        .drop("rn")
    )
    delete_ids = latest.filter("op = 'D'").select("order_id")
    survivors = current_df.join(delete_ids, "order_id", "left_anti")
    upserts = latest.filter("op IN ('I', 'U')").select(
        "order_id", "status", "total_amount", "sequence_no"
    )

    unchanged = survivors.join(upserts.select("order_id"), "order_id", "left_anti").select(
        "order_id", "status", "total_amount"
    ).withColumn("sequence_no", F.lit(0).cast("long"))
    return unchanged.unionByName(upserts).orderBy("order_id")


current = seed_orders.select("order_id", "status", "total_amount")
snapshot_once = apply_cdc(current, changes)
snapshot_twice = apply_cdc(snapshot_once.select("order_id", "status", "total_amount"), changes)

assert snapshot_once.collect() == snapshot_twice.collect()
assert snapshot_once.filter(F.col("order_id") == seed_ids[1]).count() == 0
assert snapshot_once.filter(F.col("order_id") == "ord_999999").count() == 1
snapshot_once.show(truncate=False)
print("PASS: the CDC snapshot is idempotent for this event set.")


+----------+-------+------------+-----------+
|order_id  |status |total_amount|sequence_no|
+----------+-------+------------+-----------+
|ord_000001|shipped|88.00       |102        |
|ord_000003|shipped|158.03      |0          |
|ord_000004|paid   |645.10      |0          |
|ord_000005|pending|17.58       |0          |
|ord_000006|pending|318.21      |0          |
|ord_000007|paid   |271.13      |0          |
|ord_000008|paid   |358.95      |0          |
|ord_000009|shipped|184.12      |0          |
|ord_000010|paid   |267.44      |0          |
|ord_000011|pending|66.99       |0          |
|ord_000012|pending|617.15      |0          |
|ord_999999|paid   |42.50       |104        |
+----------+-------+------------+-----------+

PASS: the CDC snapshot is idempotent for this event set.


- idempotent: 데이터 파이프라인을 재실행하거나 동일한 이벤트를 다시 처리해도 최종 데이터가 중복되거나 달라지지 않는 성질

## Your turn

1. Add an unknown `op = 'X'` event and quarantine it.
2. Add a source sequence checkpoint so events at or below the committed sequence
   are ignored safely.
3. Compare the physical plan of the CDC window with the final anti join.
4. Implement the same current-state result with Delta `MERGE` in Databricks.
5. Write tests for a delete of a missing key and two events sharing a timestamp.

### Interview checkpoint

Be ready to explain explicit schemas, schema evolution, deterministic deduplication,
quarantine, partition choice, late CDC events, and the exact reason the sink is
idempotent.
